# Crime Aggregates — When the Arithmetic Is Perfect and the Number Is Useless

Across fifteen states, the FBI's **property crime** rate moved **−0.6%** between 2019 and 2023.
That is a true statement, correctly computed, from a correctly assembled aggregate.

Its three largest components over the same period: burglary **−24.4%**, larceny **−9.9%**,
motor-vehicle theft **+60.7%**. A spread of **85.1 percentage points** inside a total that moved
**0.6** — a ratio of **136×**.

This is not an accounting failure. Section 5 tests whether the parts reconstruct the whole and
they do, **exactly, in 30 of 30 state-years, to 0.00%**. Unlike D11's cultural-economics leaves
and D19's election totals, there is nothing wrong with the sum. The aggregate is arithmetically
flawless and tells you almost nothing, because its components moved in opposite directions and
cancelled.

The notebook then runs one more test, and reports a result it does not believe. Correlating
coverage change against rate change for each of **10** offences finds exactly **1** significant
at 0.05 — homicide, ρ = **+0.579**, p = **0.024**. On a family of ten, the chance of at least
one such result when every true effect is zero is **40.1%**. And homicide is the offence for
which the mechanism makes least sense: a body is the hardest crime not to report, so homicide
should be the *least* coverage-sensitive offence in the table.

## Learning Objectives

1. **Decompose an aggregate before quoting its movement**, and report the component spread
   beside it.
2. **Distinguish an aggregation problem from an accounting problem.** A total that sums
   correctly can still be the wrong thing to report.
3. **Count the tests you ran**, not the ones that survived, and compute the family-wise rate.
4. **Weigh a significant result against the mechanism it implies**, and say when the mechanism
   argues against it.

## Prerequisites

- D13 (FBI CDE), which establishes this source's regime: voluntary reporting, coverage as a
  variable, and an upstream that flaps. This notebook inherits all of it and does not restate it.
- D30 (false discovery), for the family-wise argument in sections 6 and 7.
- Handbook §3 (denominators, universes and partitions) and §1.13 (voluntary reporting).

## Data Sources

**FBI Crime Data Explorer**, `api.usa.gov/crime/fbi/cde`, summarized state series, via the
`DATA_GOV_API_KEY` D13 already uses. Ten offences — two UCR aggregates and their eight
components — across fifteen named states, for calendar 2019 and calendar 2023.

**The coverage caveat from D13 applies to every figure here** and is not repeated: these are
counts from agencies that volunteered to report, the reporting set changes, and the published
rate is normalised by covered population rather than resident population.

**A note on the upstream.** During this build the CDE became unresponsive for several minutes
and then returned to 0.6-second responses with no change on this side. That is the flapping D13
documents, and it is why `CDEUpstreamUnavailable` exists as a distinct exception and why the
retrieval cell reports what it could not get rather than silently returning a shorter table.

## 1. Setup

The component lists are the FBI's own definitions of what each aggregate contains. Arson is
included in property crime here; an earlier draft of this notebook asserted the endpoint did not
serve it, which was wrong and untested — section 5 now tests the composition rather than
assuming it.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import CDEUpstreamUnavailable, fetch_fbi_cde_summarized

#: UCR aggregate -> the offences it is composed of, as the FBI defines them.
#: Arson is included: an earlier draft of this notebook asserted the endpoint did
#: not serve it, which was wrong and untested. Section 5 tests whether the parts
#: actually reconstruct the whole rather than assuming they do.
PARTS = {
    "property-crime": ["burglary", "larceny", "motor-vehicle-theft", "arson"],
    "violent-crime": ["homicide", "rape", "robbery", "aggravated-assault"],
}

#: Fifteen states, named rather than sampled, spanning region and size. A
#: national figure would need 51 live calls per offence against an upstream that
#: returns 503 intermittently, which is not a reproducible artifact (D13 §3).
STATES = ["PA", "FL", "NY", "IL", "NJ", "VA", "OH", "AZ", "MD", "TN",
          "CA", "TX", "GA", "MI", "WA"]

BASELINE, COMPARISON = "2019", "2023"
ALPHA = 0.05

OFFENCES = sorted({o for parts in PARTS.values() for o in parts} | set(PARTS))
pd.set_option("display.width", 112)
print(f"offences : {len(OFFENCES)} ({', '.join(OFFENCES)})")
print(f"states   : {len(STATES)}")
print(f"window   : calendar {BASELINE} against calendar {COMPARISON}")


offences : 10 (aggravated-assault, arson, burglary, homicide, larceny, motor-vehicle-theft, property-crime, rape, robbery, violent-crime)
states   : 15
window   : calendar 2019 against calendar 2023


## 2. Retrieval

A partial calendar year is dropped rather than averaged, because a mean over eight months is not
comparable to a mean over twelve and nothing in the output would show the difference.

In [2]:
"""Retrieve a full calendar year for each offence and state."""
records, unavailable = [], []
for offence in OFFENCES:
    for state in STATES:
        try:
            frame, _ = fetch_fbi_cde_summarized(
                state=state, offense=offence,
                from_month=f"01-{BASELINE}", to_month=f"12-{COMPARISON}",
            )
        except CDEUpstreamUnavailable:
            unavailable.append(f"{offence}/{state}")
            continue
        frame = frame.assign(calendar_year=frame["month"].str[-4:])
        for year in (BASELINE, COMPARISON):
            months = frame[frame["calendar_year"] == year]
            if len(months) != 12:
                continue          # a partial year is not a year; drop it visibly below
            records.append({
                "offence": offence, "state": state, "year": year,
                "offenses": int(months["offenses"].sum()),
                "rate": float(months["published_rate"].mean()),
                "coverage": float(months["coverage_pct"].mean()),
            })

panel = pd.DataFrame(records)
print(f"offence-state-year rows : {len(panel):,}")
print(f"complete offence-state pairs (both years): "
      f"{panel.groupby(['offence', 'state']).size().eq(2).sum()} of "
      f"{len(OFFENCES) * len(STATES)}")
print(f"unavailable this run    : {unavailable if unavailable else 'none'}")


offence-state-year rows : 300
complete offence-state pairs (both years): 150 of 150
unavailable this run    : none


## 3. What moved

In [3]:
"""How much did each offence move?"""
wide = panel.pivot_table(index=["offence", "state"], columns="year",
                         values=["rate", "coverage", "offenses"])
wide.columns = [f"{measure}_{year}" for measure, year in wide.columns]
wide = wide.dropna()
wide["pct_change"] = 100 * (wide[f"rate_{COMPARISON}"] / wide[f"rate_{BASELINE}"] - 1)
wide["coverage_change"] = wide[f"coverage_{COMPARISON}"] - wide[f"coverage_{BASELINE}"]

summary = wide.groupby(level=0)["pct_change"].agg(
    median="median", lowest="min", highest="max", states="size"
).sort_values("median")
print(f"median percentage change in the published rate, {BASELINE} to {COMPARISON}:")
print(summary.to_string(float_format=lambda v: f"{v:,.1f}"))


median percentage change in the published rate, 2019 to 2023:
                     median  lowest  highest  states
offence                                             
burglary              -24.4   -50.0      9.2      15
robbery               -17.4   -49.3     17.3      15
larceny                -9.9   -37.1     28.4      15
rape                   -9.3   -24.3     29.9      15
arson                  -8.2   -22.6    119.0      15
property-crime         -0.6   -38.4     34.8      15
violent-crime           5.3   -26.1     26.8      15
aggravated-assault     13.2   -43.1     43.5      15
homicide               18.4   -17.0     78.0      15
motor-vehicle-theft    60.7   -32.2    201.4      15


Read the extremes first. **Motor-vehicle theft rose 60.7%** at the median state and
**burglary fell 24.4%**. Between them sit the two aggregates, at **−0.6%** and **+5.3%**.

The per-state ranges are wider still: motor-vehicle theft runs from **−32.2%** to **+201.4%**
across the fifteen states, and arson — median **−8.2%** — reaches **+119.0%** somewhere. Any
single national figure for either is an average over states that did not move together.

## 4. Each aggregate against its own components

In [4]:
"""Each aggregate against the offences it is made of."""
for aggregate, parts in PARTS.items():
    available = [p for p in parts if p in summary.index]
    aggregate_move = summary.loc[aggregate, "median"]
    spread = summary.loc[available, "median"].max() - summary.loc[available, "median"].min()
    print(f"{aggregate}: {aggregate_move:+.1f}%")
    for part in available:
        row = summary.loc[part]
        print(f"   {part:<22} {row['median']:+6.1f}%   "
              f"(states range {row['lowest']:+.1f} to {row['highest']:+.1f})")
    print(f"   component spread {spread:.1f} pp against an aggregate move of "
          f"{abs(aggregate_move):.1f} pp -> {spread / abs(aggregate_move):.0f}x")
    if len(available) < len(parts):
        missing = sorted(set(parts) - set(available))
        print(f"   NOT retrieved, so the parts do not exhaust the aggregate: {missing}")
    print()


property-crime: -0.6%
   burglary                -24.4%   (states range -50.0 to +9.2)
   larceny                  -9.9%   (states range -37.1 to +28.4)
   motor-vehicle-theft     +60.7%   (states range -32.2 to +201.4)
   arson                    -8.2%   (states range -22.6 to +119.0)
   component spread 85.1 pp against an aggregate move of 0.6 pp -> 136x

violent-crime: +5.3%
   homicide                +18.4%   (states range -17.0 to +78.0)
   rape                     -9.3%   (states range -24.3 to +29.9)
   robbery                 -17.4%   (states range -49.3 to +17.3)
   aggravated-assault      +13.2%   (states range -43.1 to +43.5)
   component spread 35.8 pp against an aggregate move of 5.3 pp -> 7x



**Property crime: a 0.6-point move containing an 85.1-point spread — 136×.**

Burglary collapsed, motor-vehicle theft surged, larceny and arson fell moderately, and the total
came out flat. "Property crime was essentially unchanged" is true, and a reader who stops there
has been told that nothing happened when in fact two of the four components moved by more than
a quarter of their 2019 level in opposite directions.

**Violent crime is milder but the same shape**: a 5.3-point move over a 35.8-point spread, 7×.
Robbery fell 17.4% while homicide rose 18.4%.

**[rule]** Never publish an aggregate's movement without the spread of its components beside it.
The ratio of the two is the single number that says whether the aggregate is informative, and it
costs one groupby.

## 5. Is this an accounting problem?

The obvious suspicion, after D11 and D19, is that the aggregate does not equal its parts. Test
it rather than assume it.

In [5]:
"""Do the components add up to the aggregate the FBI publishes?"""
rows = []
for aggregate, parts in PARTS.items():
    for state in STATES:
        for year in (BASELINE, COMPARISON):
            block = panel[(panel["state"] == state) & (panel["year"] == year)]
            whole = block[block["offence"] == aggregate]["offenses"]
            components = block[block["offence"].isin(parts)]
            if whole.empty or len(components) != len(parts):
                continue
            rows.append({
                "aggregate": aggregate, "state": state, "year": year,
                "published": int(whole.iloc[0]),
                "summed": int(components["offenses"].sum()),
            })
identity = pd.DataFrame(rows)
identity["difference"] = identity["summed"] - identity["published"]
identity["pct"] = 100 * identity["difference"] / identity["published"]

print(f"state-years tested: {len(identity)}")
for aggregate, block in identity.groupby("aggregate"):
    exact = int((block["difference"] == 0).sum())
    print(f"  {aggregate:<16} exact in {exact} of {len(block)}   "
          f"median gap {block['pct'].median():+.2f}%   "
          f"range {block['pct'].min():+.2f}% to {block['pct'].max():+.2f}%")
print()
print(identity.reindex(identity["pct"].abs().sort_values(ascending=False).index)
      .head(5).to_string(index=False))


state-years tested: 60
  property-crime   exact in 30 of 30   median gap +0.00%   range +0.00% to +0.00%
  violent-crime    exact in 30 of 30   median gap +0.00%   range +0.00% to +0.00%

     aggregate state year  published  summed  difference  pct
property-crime    PA 2019     170683  170683           0  0.0
property-crime    PA 2023     200293  200293           0  0.0
 violent-crime    FL 2019      81095   81095           0  0.0
 violent-crime    FL 2023      52303   52303           0  0.0
 violent-crime    NY 2019      69533   69533           0  0.0


**Exact in 30 of 30 state-years for both aggregates, with a median gap of 0.00% and
a range of 0.00% to 0.00%.** The parts reconstruct the whole to the offence.

That is worth pausing on, because it is the opposite of what the last two notebooks found. D11's
cultural-economics leaves fell 2.58% short of their published total. D19's election returns
failed the identity in 2 of 663 state-years and in 38 of 663 after a plausible cleaning step.
Here there is **nothing wrong with the arithmetic at all**.

So this is not an accounting defect, and no amount of auditing the sum would have surfaced it.
It is an aggregation problem: the total is computed correctly over components that do not belong
together for the question being asked.

**[rule]** An identity that holds does not license the aggregate. Verifying that parts sum to a
whole says the whole was computed correctly; it says nothing about whether the whole means
anything.

## 6. One more test, run on all nine offences

D13 found that the change in coverage explains little of the change in the violent-crime rate.
A reasonable next question is whether that holds for every offence — some crimes are more
reporting-sensitive than others, so the coverage caveat might not be uniform.

Run it on all ten and report all ten.

In [6]:
"""Is the coverage caveat uniform across offence types?"""
results = []
for offence in sorted(set(summary.index)):
    block = wide.loc[offence]
    if len(block) < 4:
        continue
    rho, pval = stats.spearmanr(block["coverage_change"], block["pct_change"])
    results.append({"offence": offence, "states": len(block),
                    "rho": rho, "p": pval, "significant": pval < ALPHA})
coverage_tests = pd.DataFrame(results).sort_values("p")
print(coverage_tests.to_string(index=False, formatters={
    "rho": "{:+.3f}".format, "p": "{:.3f}".format}))

tested = len(coverage_tests)
hits = int(coverage_tests["significant"].sum())
family = 1 - (1 - ALPHA) ** tested
print()
print(f"tests run                                   : {tested}")
print(f"significant at {ALPHA}                            : {hits}")
print(f"P(at least one, if every true effect is zero): {family:.1%}")


            offence  states    rho     p  significant
           homicide      15 +0.579 0.024         True
              arson      15 +0.382 0.160        False
               rape      15 +0.257 0.355        False
           burglary      15 -0.204 0.467        False
            robbery      15 -0.204 0.467        False
      violent-crime      15 +0.189 0.499        False
 aggravated-assault      15 +0.161 0.567        False
     property-crime      15 -0.104 0.713        False
            larceny      15 +0.032 0.909        False
motor-vehicle-theft      15 +0.025 0.930        False

tests run                                   : 10
significant at 0.05                            : 1
P(at least one, if every true effect is zero): 40.1%


**One of ten clears 0.05: homicide, ρ = +0.579, p = 0.024.**

Before interpreting it, note what the table above cost. Ten tests were run at α = 0.05. If every
true effect were zero, the chance of at least one result this significant is
1 − 0.95¹⁰ = **40.1%**. A single significant result out of ten is not evidence of anything; it is
the expected yield of the procedure.

This is the D30 lesson arriving in live data rather than a simulation, and the defence is the
same: **report the denominator of the search**. A notebook that ran ten tests and showed one is
not reporting a finding, it is reporting a selection.

## 7. And the mechanism argues against it

Multiplicity alone would leave the homicide result merely unproven. There is a second reason to
disbelieve it, and it is stronger.

In [7]:
"""The one significant result, against what theory expects."""
top = coverage_tests.iloc[0]
print(f"most significant: {top['offence']} rho={top['rho']:+.3f} p={top['p']:.3f} "
      f"on {int(top['states'])} states")
print()
print("A positive rho means the states that GAINED reporting coverage are the states")
print("whose measured rate rose most -- which is what you would expect for an offence")
print("that often goes unreported, because new agencies bring previously unseen")
print("incidents with them.")
print()
print("Homicide is the offence for which that mechanism is weakest. A body is the")
print("single hardest crime to not report, so homicide should be the LEAST")
print("coverage-sensitive offence in the table, not the most.")
print()
print("So the one result that clears the threshold has the wrong subject and, on a")
print("family of this size, is about as likely to appear by chance as not. It is")
print("reported here and is not a finding.")


most significant: homicide rho=+0.579 p=0.024 on 15 states

A positive rho means the states that GAINED reporting coverage are the states
whose measured rate rose most -- which is what you would expect for an offence
that often goes unreported, because new agencies bring previously unseen
incidents with them.

Homicide is the offence for which that mechanism is weakest. A body is the
single hardest crime to not report, so homicide should be the LEAST
coverage-sensitive offence in the table, not the most.

So the one result that clears the threshold has the wrong subject and, on a
family of this size, is about as likely to appear by chance as not. It is
reported here and is not a finding.


The sign is what a coverage effect should look like — states that gained reporting
agencies saw measured rates rise, because new agencies bring previously unseen incidents.

But that mechanism works through **crimes that go unreported when no agency is reporting**.
Homicide is the offence where it is weakest: a body is the single hardest crime to conceal from
the statistical system, which is why homicide counts are treated as the most complete series in
American crime statistics.

So the one result that survived the threshold is on the offence where the proposed mechanism has
the least room to operate. Between a 40.1% family-wise rate and a mechanism pointing the wrong
way, the reasonable reading is that this is noise.

**[rule]** A significant result whose mechanism runs backwards is weaker evidence than a
non-significant one whose mechanism is sound. Report it, say why you do not believe it, and do
not quietly drop it — a reader who finds it later without your reasoning will believe it.

## Limitations

1. **Everything D13 says about this source applies here** — voluntary reporting, a changing
   agency set, and rates normalised by covered population. None of it is repeated and none of it
   is resolved.

2. **Fifteen named states, not a sample.** The medians are medians over those fifteen and are not
   national estimates. No weighting by population is applied.

3. **Two endpoints of a period, not a trend.** 2019 against 2023 says nothing about the path
   between them, which for several of these offences was not monotonic.

4. **The 2019–2023 window spans the pandemic**, which moved reporting behaviour, policing and
   opportunity simultaneously. Nothing here separates those.

5. **The component spread is a descriptive statistic, not a test.** It has no sampling
   distribution attached, because these are administrative counts rather than estimates.

6. **Section 6's ten tests are themselves a chosen family.** Ten offences on fifteen states over
   one window; a different set would give a different family-wise rate, and that rate is a
   property of the search rather than of crime.

7. **No causal claim is made anywhere.** Motor-vehicle theft rose; this notebook does not say
   why, and the widely-reported explanation for part of that rise is not tested here.

**On vintage.** The FBI CDE serves current data with a moving max_data_date and no prior vintage, so the month range names a reference period rather than a vintage. This notebook therefore records `vintage_pinned: false` (handbook §5.2); the payload hash and retrieval time in its provenance record are what a later reader has to compare against.

## Key Takeaways

1. **Property crime moved −0.6% while its components spread 85.1 points** — 136×. Violent crime
   moved +5.3% over a 35.8-point spread, 7×.

2. **The arithmetic is perfect.** Components reconstruct both aggregates exactly, in 30 of 30
   state-years, to 0.00%.

3. **An accounting audit would have found nothing**, because there is nothing wrong with the sum.
   This is an aggregation problem, not an accounting one.

4. **Report the component spread beside every aggregate movement.** The ratio is what says
   whether the aggregate is informative.

5. **Ten tests, one significant, 40.1% family-wise.** Report the denominator of the search.

6. **The one significant result has a backwards mechanism.** Homicide should be the least
   coverage-sensitive offence, not the most.

7. **Report a result you disbelieve, with the reason.** A reader who finds it without your
   reasoning will believe it.

## Exercises

1. **Find the ratio's breaking point.** Compute the aggregate-move-to-component-spread ratio for
   every pair of adjacent years from 2019 to 2023. Is 2019→2023 unusual, or is property crime
   always this uninformative?

2. **Weight it.** The medians here are unweighted across fifteen states. Recompute
   population-weighted and report which conclusions change and which do not.

3. **Run the family properly.** Apply Benjamini–Hochberg to section 6's ten p-values and report
   what survives. Then say whether that changes your reading of the homicide result, and why.

4. **Test the mechanism directly.** If coverage effects work through unreported crime, the
   coverage sensitivity should be ordered by how reportable an offence is. Rank the ten offences
   by an a-priori reportability argument, then correlate that ranking against the ρ column.

5. **Break the aggregate deliberately.** Construct the smallest change to one component that
   would move property crime by 5% at the median state, and state how large that change is
   relative to the component's own 2019 level.

6. **Write the sentence.** In one sentence a wire service would print, report what happened to
   property crime in these fifteen states between 2019 and 2023.

## References

- Federal Bureau of Investigation. *Crime Data Explorer.* https://cde.ucr.cgis.fbi.gov/
- Federal Bureau of Investigation. *UCR offence definitions* — the composition of the violent
  crime and property crime aggregates.
  https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/violent-crime
- Benjamini, Y. and Hochberg, Y. (1995). "Controlling the False Discovery Rate." *JRSS B* 57(1).
- D13 (`07-justice-equity/D13_crime_and_public_safety`) — the regime this notebook inherits.
- D30 (`02-health-housing/D30_subjective_wellbeing`) — the false-discovery argument, measured.
- Handbook §1.13 (voluntary reporting), §3 (denominators and partitions).